In [1]:
from glob import glob
import pandas as pd
DATA_DIR = "/DATA/disk2/yuhang/.cache/modelscope/datasets/gongjy/minimind_dataset"
from tqdm import tqdm
import json
import re
import random


In [2]:
# 读取DATA_DIR中的所有jsonl文件并分析列信息
def analyze_jsonl_files(data_dir):
    """
    分析指定目录下所有jsonl文件的列信息
    
    Args:
        data_dir: 数据目录路径
    """
    # 查找所有jsonl文件
    jsonl_files = glob(f"{data_dir}/**/*.jsonl", recursive=True)
    
    if not jsonl_files:
        print(f"在目录 {data_dir} 中没有找到任何jsonl文件")
        return
    
    print(f"找到 {len(jsonl_files)} 个jsonl文件:")
    for file in jsonl_files:
        print(f"  - {file}")
    
    print("\n" + "="*50)
    
    # 分析每个文件的列信息
    for file_path in tqdm(jsonl_files, desc="分析文件"):
        print(f"\n文件: {file_path}")
        
        try:
            # 读取文件的前几行来分析结构
            sample_data = []
            with open(file_path, 'r', encoding='utf-8') as f:
                for i, line in enumerate(f):
                    if i >= 5:  # 只读取前5行作为样本
                        break
                    try:
                        data = json.loads(line.strip())
                        sample_data.append(data)
                    except json.JSONDecodeError as e:
                        print(f"  警告: 第{i+1}行JSON解析错误: {e}")
                        continue
            
            if not sample_data:
                print("  错误: 无法读取有效的JSON数据")
                continue
            
            # 分析列信息
            all_keys = set()
            for data in sample_data:
                if isinstance(data, dict):
                    all_keys.update(data.keys())
            
            print(f"  总行数估计: {sum(1 for _ in open(file_path, 'r', encoding='utf-8'))}")
            print(f"  列数: {len(all_keys)}")
            print(f"  列名: {list(all_keys)}")
            
            # 显示每列的数据类型和样本值
            for key in sorted(all_keys):
                values = []
                types = set()
                for data in sample_data:
                    if key in data:
                        value = data[key]
                        values.append(value)
                        types.add(type(value).__name__)
                
                # 显示样本值（截断长文本）
                sample_value = str(values[0]) if values else "None"
                if len(sample_value) > 100:
                    sample_value = sample_value[:100] + "..."
                
                print(f"    {key}: 类型={list(types)}, 样本值='{sample_value}'")
        
        except Exception as e:
            print(f"  错误: 处理文件时出现异常: {e}")
        
        print("-" * 30)

# 执行分析
analyze_jsonl_files(DATA_DIR)


在目录 /DATA/disk2/yuhang/.cache/modelscope/datasets/gongjy/minimind_dataset 中没有找到任何jsonl文件


In [2]:
import json
import os
from tqdm import tqdm

def convert_to_shared_gpt_format(input_dir, output_file):
    """
    将jsonl格式的对话数据转换为shared-gpt格式
    
    Args:
        input_dir: 输入目录路径，包含jsonl文件
        output_file: 输出文件路径
    """
    print(f"开始转换数据格式...")
    print(f"输入目录: {input_dir}")
    print(f"输出文件: {output_file}")
    
    # 获取所有jsonl文件
    jsonl_files = [f for f in os.listdir(input_dir) if f.endswith('.jsonl')]
    print(f"找到 {len(jsonl_files)} 个jsonl文件")
    
    converted_data = []
    total_conversations = 0
    
    # 处理每个jsonl文件
    for file_name in tqdm(jsonl_files, desc="处理文件"):
        file_path = os.path.join(input_dir, file_name)
        print(f"\n处理文件: {file_name}")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    try:
                        # 解析JSON数据
                        data = json.loads(line.strip())
                        
                        # 检查是否包含conversations字段
                        if 'conversations' not in data:
                            print(f"  警告: 第{line_num}行缺少conversations字段")
                            continue
                        
                        conversations = data['conversations']
                        if not isinstance(conversations, list):
                            print(f"  警告: 第{line_num}行conversations不是列表格式")
                            continue
                        
                        # 转换为shared-gpt格式
                        shared_gpt_item = {
                            "conversations": []
                        }
                        
                        # 处理对话内容
                        for conv in conversations:
                            if isinstance(conv, dict) and 'role' in conv and 'content' in conv:
                                # 映射角色名称
                                role = conv['role']
                                if role == 'user':
                                    mapped_role = 'human'
                                elif role == 'assistant':
                                    mapped_role = 'gpt'
                                else:
                                    mapped_role = role  # 保持原有角色名
                                
                                shared_gpt_item["conversations"].append({
                                    "from": mapped_role,
                                    "value": conv['content']
                                })
                            else:
                                print(f"  警告: 第{line_num}行对话格式不正确: {conv}")
                        
                        # 只添加有效的对话
                        if shared_gpt_item["conversations"]:
                            converted_data.append(shared_gpt_item)
                            total_conversations += 1
                        
                    except json.JSONDecodeError as e:
                        print(f"  警告: 第{line_num}行JSON解析错误: {e}")
                        continue
                    except Exception as e:
                        print(f"  警告: 第{line_num}行处理错误: {e}")
                        continue
        
        except Exception as e:
            print(f"  错误: 处理文件 {file_name} 时出现异常: {e}")
            continue
    
    # 保存转换后的数据
    print(f"\n开始保存转换后的数据...")
    print(f"总共转换了 {total_conversations} 个对话")
    
    try:
        # 确保输出目录存在
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        
        # 保存为jsonl格式
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in tqdm(converted_data, desc="保存数据"):
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"✅ 数据转换完成!")
        print(f"输出文件: {output_file}")
        print(f"转换的对话数量: {total_conversations}")
        
        # 显示转换后的数据样本
        if converted_data:
            print(f"\n转换后的数据样本:")
            sample = converted_data[0]
            print(json.dumps(sample, ensure_ascii=False, indent=2)[:500] + "...")
        
    except Exception as e:
        print(f"❌ 保存文件时出现错误: {e}")

# 设置输入和输出路径
INPUT_DIR = DATA_DIR  # 使用之前定义的数据目录
OUTPUT_FILE = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl"

# 执行转换
convert_to_shared_gpt_format(INPUT_DIR, OUTPUT_FILE)


开始转换数据格式...
输入目录: /DATA/disk2/yuhang/.cache/modelscope/datasets/gongjy/minimind_dataset
输出文件: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl
找到 2 个jsonl文件


处理文件:   0%|          | 0/2 [00:00<?, ?it/s]


处理文件: sft_2048.jsonl


处理文件:   0%|          | 0/2 [00:12<?, ?it/s]


KeyboardInterrupt: 

In [3]:
# 读取并分析转换后的数据集
import re
import json
from tqdm import tqdm
from collections import Counter
from langdetect import detect, DetectorFactory, LangDetectException

# 设置随机种子保证结果一致性
DetectorFactory.seed = 0

def analyze_language_distribution(jsonl_file):
    """
    使用langdetect分析JSONL数据集中中英文数据的比例
    
    Args:
        jsonl_file (str): JSONL文件路径
    """
    print(f"开始分析数据集: {jsonl_file}")
    
    # 统计变量
    total_conversations = 0
    language_stats = {
        'zh': 0,    # 中文对话
        'en': 0,    # 英文对话
        'mixed': 0, # 中英混合对话
        'other': 0  # 其他语言
    }
    
    try:
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(tqdm(f, desc="分析语言分布"), 1):
                try:
                    data = json.loads(line.strip())
                    conversations = data.get('conversations', [])
                    if not conversations:
                        continue
                    
                    total_conversations += 1
                    lang_counts = Counter()
                    
                    # 分析每个对话轮次
                    for conv in conversations:
                        text = conv.get('value', '')[:500]  # 截取前500字符提高检测效率
                        if not text.strip():
                            continue
                        
                        try:
                            lang = detect(text)
                            lang_counts[lang] += 1
                        except LangDetectException:
                            lang_counts['unknown'] += 1
                    
                    # 判断对话语言类型
                    main_langs = [lang for lang, _ in lang_counts.most_common(2)]
                    if 'zh-cn' in main_langs or 'zh-tw' in main_langs:
                        if 'en' in main_langs:
                            language_stats['mixed'] += 1
                        else:
                            language_stats['zh'] += 1
                    elif 'en' in main_langs:
                        language_stats['en'] += 1
                    elif len(main_langs) > 0:
                        language_stats['other'] += 1
                    else:
                        language_stats['other'] += 1
                        
                except json.JSONDecodeError as e:
                    print(f"  警告: 第{line_num}行JSON解析错误: {e}")
                except Exception as e:
                    print(f"  警告: 第{line_num}行处理错误: {e}")
    
        # 输出统计结果
        print(f"\n📊 数据集语言分布统计结果:")
        print("=" * 50)
        print(f"总对话数量: {total_conversations}")
        
        if total_conversations > 0:
            zh_percent = (language_stats['zh'] / total_conversations) * 100
            en_percent = (language_stats['en'] / total_conversations) * 100
            mixed_percent = (language_stats['mixed'] / total_conversations) * 100
            other_percent = (language_stats['other'] / total_conversations) * 100
            
            print(f"🇨🇳 中文对话: {language_stats['zh']} ({zh_percent:.1f}%)")
            print(f"🇺🇸 英文对话: {language_stats['en']} ({en_percent:.1f}%)")
            print(f"🌐 中英混合: {language_stats['mixed']} ({mixed_percent:.1f}%)")
            print(f"❓ 其他语言: {language_stats['other']} ({other_percent:.1f}%)")
            
    except FileNotFoundError:
        print(f"❌ 错误: 找不到文件 {jsonl_file}")
    except Exception as e:
        print(f"❌ 错误: 分析文件时出现异常: {e}")

# 分析转换后的数据集
OUTPUT_FILE = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl"
analyze_language_distribution(OUTPUT_FILE)


开始分析数据集: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl


分析语言分布: 7915003it [7:07:17, 308.73it/s]


KeyboardInterrupt: 

In [4]:
import json
import random
from tqdm import tqdm

def random_sample_data(input_file, output_file, target_count=200000):
    """
    从输入文件中随机采样指定数量的对话数据并保存到输出文件
    
    Args:
        input_file (str): 输入文件路径
        output_file (str): 输出文件路径  
        target_count (int): 目标采样数量，默认200K
    """
    print(f"🎲 开始随机采样数据...")
    print(f"📁 输入文件: {input_file}")
    print(f"📁 输出文件: {output_file}")
    print(f"🎯 目标数量: {target_count:,} 条")
    print("=" * 60)
    
    # 第一步：读取所有数据
    all_data = []
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            print("📊 正在读取所有数据...")
            
            for line_num, line in enumerate(tqdm(f, desc="读取数据"), 1):
                line = line.strip()
                if not line:
                    continue
                    
                try:
                    # 解析JSON数据
                    data = json.loads(line)
                    
                    # 检查数据格式是否正确
                    if 'conversations' in data and isinstance(data['conversations'], list):
                        all_data.append(data)
                        
                except json.JSONDecodeError as e:
                    print(f"  警告: 第{line_num}行JSON解析错误: {e}")
                except Exception as e:
                    print(f"  警告: 第{line_num}行处理错误: {e}")
        
        print(f"📄 成功读取数据总数: {len(all_data):,} 条")
        
        # 第二步：随机采样
        if len(all_data) <= target_count:
            print(f"⚠️  数据总数({len(all_data):,})小于等于目标数量({target_count:,})，将使用全部数据")
            sampled_data = all_data
        else:
            print(f"🎲 正在从 {len(all_data):,} 条数据中随机采样 {target_count:,} 条...")
            sampled_data = random.sample(all_data, target_count)
        
        # 第三步：保存采样结果
        print(f"💾 正在保存采样结果到: {output_file}")
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for data in tqdm(sampled_data, desc="保存数据"):
                f.write(json.dumps(data, ensure_ascii=False) + '\n')
        
        print(f"✅ 随机采样完成！")
        print(f"📊 最终保存数据量: {len(sampled_data):,} 条")
        print("=" * 60)
        
    except FileNotFoundError:
        print(f"❌ 错误: 找不到输入文件 {input_file}")
    except Exception as e:
        print(f"❌ 错误: 处理文件时出现异常: {e}")

# 执行随机采样
INPUT_FILE = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl"
OUTPUT_FILE = "/DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/random_sample_200k.jsonl"
random_sample_data(INPUT_FILE, OUTPUT_FILE, target_count=200000)


🎲 开始随机采样数据...
📁 输入文件: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/shared_gpt_format.jsonl
📁 输出文件: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/random_sample_200k.jsonl
🎯 目标数量: 200,000 条
📊 正在读取所有数据...


读取数据: 9587027it [01:35, 100550.31it/s]


📄 成功读取数据总数: 9,587,027 条
🎲 正在从 9,587,027 条数据中随机采样 200,000 条...
💾 正在保存采样结果到: /DATA/disk2/yuhang/.cache/bit_brain_data/sft_llamafactory/minimind/random_sample_200k.jsonl


保存数据: 100%|██████████| 200000/200000 [00:02<00:00, 96188.78it/s]


✅ 随机采样完成！
📊 最终保存数据量: 200,000 条


In [ ]:
# 设置随机种子以确保结果可复现
random.seed(42)

# 执行随机采样
random_sample_data(INPUT_FILE, OUTPUT_FILE, target_count=200000)

import json
import random
from langdetect import detect, LangDetectError
from tqdm import tqdm

def filter_english_data(input_file, output_file, target_count=200000):
    """
    从输入文件中筛选出英文对话数据并保存到输出文件
    
    Args:
        input_file (str): 输入文件路径
        output_file (str): 输出文件路径  
        target_count (int): 目标筛选数量，默认200K
    """
    print(f"🔍 开始筛选英文数据...")
    print(f"📁 输入文件: {input_file}")
    print(f"📁 输出文件: {output_file}")
    print(f"🎯 目标数量: {target_count:,} 条")
    print("=" * 60)
    
    english_data = []
    total_processed = 0
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            # 首先统计总行数用于进度条
            print("📊 正在统计文件总行数...")
            total_lines = sum(1 for _ in f)
            f.seek(0)  # 重置文件指针
            
            print(f"📄 文件总行数: {total_lines:,}")
            
            # 使用进度条处理每一行
            for line_num, line in enumerate(tqdm(f, total=total_lines, desc="筛选英文数据"), 1):
                if len(english_data) >= target_count:
                    print(f"✅ 已达到目标数量 {target_count:,} 条，停止筛选")
                    break
                    
                line = line.strip()
                if not line:
                    continue
                    
                try:
                    # 解析JSON数据
                    data = json.loads(line)
                    total_processed += 1
                    
                    # 检查数据格式
                    if 'conversations' not in data:
                        continue
                        
                    conversations = data['conversations']
                    if not isinstance(conversations, list) or len(conversations) == 0:
                        continue
                    
                    # 检测对话语言
                    is_english = True
                    english_text_count = 0
                    total_text_count = 0
                    
                    for conv in conversations:
                        if isinstance(conv, dict) and 'value' in conv:
                            text = conv['value'].strip()
                            if len(text) > 10:  # 只检测长度大于10的文本
                                total_text_count += 1
                                try:
                                    detected_lang = detect(text)
                                    if detected_lang == 'en':
                                        english_text_count += 1
                                except LangDetectError:
                                    # 语言检测失败，跳过这段文本
                                    pass
                    
                    # 判断是否为英文对话（至少70%的文本被识别为英文）
                    if total_text_count > 0:
                        english_ratio = english_text_count / total_text_count
                        if english_ratio >= 0.7:
                            english_data.append(data)
                            
                            # 每筛选出1000条数据显示一次进度
                            if len(english_data) % 1000 == 0:
                                print(f"  ✨ 已筛选出 {len(english_data):,} 条英文数据")
                    
                except json.JSONDecodeError as e:
                    print(f"  ⚠️ 第{line_num}行JSON解析错误: {e}")
                    continue
                except Exception as e:
                    print(f"  ⚠️ 第{line_num}行处理错误: {e}")
                    continue
        
        print(f"\n📊 筛选完成统计:")
        print("=" * 40)
        print(f"总处理行数: {total_processed:,}")
        print(f"筛选出英文数据: {len(english_data):,} 条")
        print(f"筛选成功率: {(len(english_data)/total_processed)*100:.2f}%")
        
        # 如果筛选出的数据超过目标数量，随机采样
        if len(english_data) > target_count:
            print(f"🎲 数据量超过目标，随机采样 {target_count:,} 条...")
            english_data = random.sample(english_data, target_count)
        
        # 保存筛选结果
        print(f"💾 正在保存到文件: {output_file}")
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in tqdm(english_data, desc="保存数据"):
                f.write(json.dumps(item, ensure_ascii=False) + '\n')
        
        print(f"✅ 筛选完成！")
        print(f"📁 输出文件: {output_file}")
        print(f"📊 最终保存数据量: {len(english_data):,} 条")
        
    except FileNotFoundError:
        print(f"❌ 错误: 找不到输入文件 {input_file}")
    except Exception as e:
        print(f"❌ 错误: 处理文件时出现异常: {e}")

# 执行英文数据筛选
INPUT_FILE = "/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input/shared_gpt_format.jsonl"
OUTPUT_FILE = "/DATA/disk2/yuhang/.cache/steel_dataset/sft_data/llamafactory_input/deepctrl_en_200k.jsonl"

# 设置随机种子以确保结果可复现
random.seed(42)

# 开始筛选
filter_english_data(INPUT_FILE, OUTPUT_FILE, target_count=200000)